# Solar Filament Segmentation 2026 — reproduce the final submission

This notebook regenerates our final submission (**public LB PQ 0.40, out-of-fold PQ 0.451**) from the
weights committed in `weights/`, **without any training**. It needs only this repository, the
packages in `requirements.txt`, and the competition's test images.

**Pipeline** (details in the report, `report/main.pdf`, and in `src/reproduce.py`):

1. **20 YOLO11s fold detectors** (`weights/detectors/v09|v10|v11|v13/fold{0-4}_yolo.pt`) predict boxes on each
   native 2048×2048 test image (input size 1280, conf ≥ 0.05, class-agnostic NMS 0.5).
   A box's score is its confidence × its *annotator-agreement class* weight (1 / 0.75 / 0.5).
2. **Weighted boxes fusion** merges the 20 box sets of an image (IoU 0.55; score = Σ best score / 20).
3. **5 crop refiners** (`weights/refiners/fold{0-4}_refiner.pt`, U-Net/ResNet34 on [image, CLAHE, box seed])
   draw a mask inside every fused box at native resolution, averaged with 4-flip TTA.
4. **Instances**: keep boxes with score ≥ 0.25, mask probability ≥ 0.6, components touching the box seed,
   ≥ 100 px, no two instances share a pixel. One RLE row per instance → `submission.csv`.

Training code: `src/` (all modules), assembled into the Kaggle training kernel by
`scripts/build_notebook.py` → `notebooks/submission.ipynb`. The experiment log is `PORTING.md`.

## 1. Setup
Set `DATA_ROOT` to the competition data folder (the one containing `test/test_images/`).

In [ ]:
import sys, os, time
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

DATA_ROOT = Path(os.environ.get("FILAMENT_DATA", REPO / "data" / "MAGFiLO_1.0_Kaggle_2026"))
assert (DATA_ROOT / "test" / "test_images").is_dir(), f"test images not found under {DATA_ROOT}"
print("repo:", REPO); print("data:", DATA_ROOT)

## 2. Verify the committed weights
Every weight file is checked against `weights/SHA256SUMS`.

In [ ]:
from reproduce import verify_weights, reproduce, compare_submissions, FINAL_KNOBS, DETECTOR_RUNS
print("verified", verify_weights(), "weight files; detector runs:", DETECTOR_RUNS, "; knobs:", FINAL_KNOBS)

## 3. Quick check on 3 test images (~1-2 min on CPU)

In [ ]:
df3, crops3 = reproduce(DATA_ROOT, REPO / "results" / "quick_check.csv", limit=3, log_every=1)
df3.head()

In [ ]:
# visualise the predicted filaments on the first image
import cv2, numpy as np, matplotlib.pyplot as plt
from rle import rle_to_mask
from data import test_img_dir
img_id = df3.filament_id.str.rsplit("_", n=1).str[0].iloc[0]
img = cv2.imread(str(test_img_dir() / f"{img_id}.jpeg"), cv2.IMREAD_GRAYSCALE)
rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
for r in df3[df3.filament_id.str.startswith(img_id)].segmentation_rle:
    cs, _ = cv2.findContours(rle_to_mask(r), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(rgb, cs, -1, (255, 40, 20), 4)
plt.figure(figsize=(8, 8)); plt.imshow(rgb); plt.axis("off"); plt.title(f"{img_id}: predicted filaments"); plt.show()

## 4. Full test set (180 images, ~1.7 h on a laptop CPU, minutes on a GPU)

In [ ]:
t = time.time()
df, crops = reproduce(DATA_ROOT, REPO / "results" / "submission_reproduced.csv")
print(f"{len(df)} rows in {(time.time() - t) / 60:.1f} min")

## 5. Compare with the file we submitted
`results/submission_final.csv` is the exact CSV uploaded to Kaggle. On our machine all 1,218 instances are bit-identical.

In [ ]:
compare_submissions(REPO / "results" / "submission_reproduced.csv", REPO / "results" / "submission_final.csv")